<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_01_intraday_data_preparation/stage_01_intraday_data_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **stage_01_intraday_data_preparation**



## Resumen

Esta notebook realiza la **preparación inicial del dataset intradía del MNQ (Micro E-mini Nasdaq 100)**.  
El objetivo es construir un dataset limpio, consistente y estructurado que servirá como base para la ingeniería de factores y el entrenamiento de modelos.

0. **Configuración del entorno**
   - Clonado del repositorio y montaje de Google Drive.
   - Instalación e importación de librerías necesarias.

1. **Fuente de datos**
   - Datos históricos intradía del MNQ (OHLCV, frecuencia de 1 minuto) exportados desde NinjaTrader.
   - Archivos originales en formato `.txt`, en zona horaria UTC.

2. **Generación del dataset**
   - Unificación de todos los archivos `.txt` en un único DataFrame.
   - Asignación de nombres de columnas: `open`, `high`, `low`, `close`, `volume`.
   - Conversión de la columna `datetime` a índice temporal.

3. **Filtrado**
   - Conserva solo **días hábiles bursátiles** (se eliminan fines de semana y feriados de mercado de EE.UU.).
   - Conversión de marcas de tiempo de **UTC → US/Eastern**.
   - Filtrado de **horario de mercado** (09:30–16:00) más pre-market (desde 08:30).

4. **Validación de registros diarios**
   - Verificación de que cada día contenga la cantidad esperada de registros minuto a minuto.
   - Detección y eliminación de días incompletos o con irregularidades.

5. **Chequeo de continuidad temporal**
   - Confirmación de que los datos intradía estén en intervalos consecutivos de 1 minuto, sin gaps.

6. **Dataset final**
   - Guardado del dataset limpio en formato `.parquet` dentro de Google Drive.

---

**Resultado:** Un dataset intradía del MNQ completamente limpio y estandarizado, listo para la ingeniería de factores y el modelado.

## 0. Configuración del Entorno

### 0.1. Acceso a Drive


In [1]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Mounted at /content/drive


### 0.2. Instalación de librerías

In [2]:
import sys
!{sys.executable} -m pip install -q pandas_market_calendars
print("✅ Librería instalada: pandas_market_calendars")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.4/131.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 4.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ibis-framework 9.5.0 requires toolz<1,>=0.11, but you have toolz 1.1.0 which is incompatible.
✅ Librería instalada: pandas_market_calendars


### 0.3. Importación de librerías

In [3]:
from __future__ import annotations  # permite type hints modernos (Python < 3.11)

# Utilidades generales
from datetime import datetime, timedelta
from io import StringIO
from pathlib import Path
import glob
import json
import os
import warnings

from dataclasses import dataclass
from typing import Any, Dict, Optional, Tuple

warnings.filterwarnings("ignore")

# Manejo y procesamiento de datos
import pandas as pd
from tabulate import tabulate

# Calendario de mercados y requests
import pandas_market_calendars as mcal
import requests

### 0.4. Rutas de archivos de entrada y salida

In [4]:
RAW_DIR = Path(os.environ.get("RAW_DIR", "data/raw/mnq_raw.parquet"))
OUT_PARQUET = Path(os.environ.get("OUT_PARQUET", "data/processed/mnq_intraday.parquet"))
OUT_SUMMARY = Path(os.environ.get("OUT_SUMMARY", "reports/stage_01_dataset_prep_summary.json"))

DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

# PARA EL NOTEBOOK:
RAW_DIR = DRIVE_DIR / RAW_DIR
OUT_PARQUET = DRIVE_DIR / OUT_PARQUET
OUT_SUMMARY = DRIVE_DIR / OUT_SUMMARY

# **1. Carga del dataset crudo `mnq_raw`**

In [5]:
def load_raw_dataset():
    os.path.exists(RAW_DIR)
    print("Archivo encontrado en disco. Cargando dataset local...")
    mnq_raw = pd.read_parquet(RAW_DIR)
    return mnq_raw

In [6]:
mnq_raw = load_raw_dataset()

Archivo encontrado en disco. Cargando dataset local...


In [7]:
mnq_raw

,open,high,low,close,volume
datetime,,,,,
2019-12-23 03:01:00,8718.50,8718.75,8718.50,8718.50,9
2019-12-23 03:02:00,8718.25,8718.25,8718.00,8718.25,14
2019-12-23 03:03:00,8718.25,8718.50,8718.00,8718.25,74
2019-12-23 03:04:00,8718.25,8719.00,8718.25,8718.50,10
2019-12-23 03:05:00,8718.50,8719.00,8718.50,8719.00,6
...,...,...,...,...,...
2025-06-15 23:59:00,21687.25,21687.75,21682.75,21686.50,216
2025-06-16 00:00:00,21686.50,21691.75,21686.00,21687.25,103
2025-06-16 00:01:00,21687.25,21694.00,21684.50,21688.00,266


In [8]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Optional, Tuple

import pandas as pd


def mnq_dataset_info(
    df: pd.DataFrame,
    *,
    name: str = "mnq_raw",
    tz_assume_if_naive: Optional[str] = None,  # ej: "UTC" o "America/New_York"
    day_def: str = "calendar",  # "calendar" (fecha calendario) o "trading" (días con datos)
) -> Dict[str, Any]:
    """
    Resume un dataset OHLCV con DatetimeIndex (ideal para mnq_raw).

    - Si el índice es tz-naive:
        - Si tz_assume_if_naive != None, lo localiza a esa tz.
        - Si no, reporta "tz-naive" (no se puede afirmar horario UTC).
    - Devuelve dict con métricas principales (y lo imprime bonito si se desea).
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError(f"{name}: se requiere DatetimeIndex, recibido: {type(df.index)}")

    idx = df.index

    # --- timezone / UTC info ---
    tzinfo = idx.tz
    if tzinfo is None:
        tz_status = "tz-naive (sin zona horaria)"
        if tz_assume_if_naive:
            idx = idx.tz_localize(tz_assume_if_naive)
            tzinfo = idx.tz
            tz_status = f"localizado como {tzinfo}"
    else:
        tz_status = f"{tzinfo}"

    # --- rango temporal ---
    ts_min = idx.min()
    ts_max = idx.max()

    first_day = ts_min.date()
    last_day = ts_max.date()

    # --- días ---
    if day_def == "calendar":
        total_days = (pd.Timestamp(last_day) - pd.Timestamp(first_day)).days + 1
    elif day_def == "trading":
        total_days = idx.normalize().nunique()
    else:
        raise ValueError("day_def debe ser 'calendar' o 'trading'")

    # --- columnas ---
    columns = list(df.columns)

    # --- checks útiles ---
    n_rows = len(df)
    n_cols = df.shape[1]
    n_missing = int(df.isna().sum().sum())
    missing_by_col = df.isna().sum().to_dict()
    dup_index = int(idx.duplicated().sum())
    is_monotonic = bool(idx.is_monotonic_increasing)

    # Frecuencia estimada (puede fallar si hay huecos grandes)
    freq = pd.infer_freq(idx[: min(50000, len(idx))])  # muestra grande pero acotada

    # Cobertura por día (min/max de hora del día, en tz del índice)
    # (Útil para ver si es 24/7 o horario de sesión)
    tod = pd.Series(idx.time)
    # Convertimos time a minutos del día para resumen robusto
    tod_minutes = pd.Series([t.hour * 60 + t.minute for t in tod])
    typical_minute_min = int(tod_minutes.min())
    typical_minute_max = int(tod_minutes.max())

    # Rango promedio de filas por día (sólo días con datos)
    rows_per_day = df.groupby(idx.normalize()).size()
    rows_per_day_stats = {
        "days_with_data": int(rows_per_day.shape[0]),
        "rows_per_day_min": int(rows_per_day.min()),
        "rows_per_day_p50": float(rows_per_day.median()),
        "rows_per_day_max": int(rows_per_day.max()),
    }

    # Si hay tz, también mostramos rango en UTC
    if tzinfo is not None:
        ts_min_utc = ts_min.tz_convert("UTC")
        ts_max_utc = ts_max.tz_convert("UTC")
        utc_range = (str(ts_min_utc), str(ts_max_utc))
        utc_note = "El índice está tz-aware; el horario UTC es inequívoco."
    else:
        utc_range = None
        utc_note = "El índice es tz-naive; no se puede asegurar si está en UTC sin suposiciones."

    info: Dict[str, Any] = {
        "name": name,
        "shape": (n_rows, n_cols),
        "columns": columns,
        "index_type": type(df.index).__name__,
        "index_tz": tz_status,
        "utc_note": utc_note,
        "datetime_min": str(ts_min),
        "datetime_max": str(ts_max),
        "first_day": str(first_day),
        "last_day": str(last_day),
        "total_days": int(total_days),
        "day_definition": day_def,
        "utc_range_if_applicable": utc_range,
        "is_index_monotonic_increasing": is_monotonic,
        "duplicated_timestamps_in_index": dup_index,
        "inferred_freq_sample": freq,
        "missing_total_cells": n_missing,
        "missing_by_col": missing_by_col,
        "rows_per_day_stats": rows_per_day_stats,
        "time_of_day_minutes_range": {
            "min_minute_of_day": typical_minute_min,
            "max_minute_of_day": typical_minute_max,
        },
    }
    return info


def print_mnq_dataset_info(info: Dict[str, Any]) -> None:
    """Imprime el dict de mnq_dataset_info de forma ordenada."""
    print(f"Dataset: {info['name']}")
    print(f"Shape: {info['shape']}")
    print(f"Columns: {info['columns']}")
    print(f"Index: {info['index_type']} | TZ: {info['index_tz']}")
    print(f"Datetime min/max: {info['datetime_min']}  ->  {info['datetime_max']}")
    print(f"First/Last day: {info['first_day']}  ->  {info['last_day']}")
    print(f"Total days ({info['day_definition']}): {info['total_days']}")
    #print(f"Inferred freq (sample): {info['inferred_freq_sample']}")
    #print(f"Index monotonic increasing: {info['is_index_monotonic_increasing']}")
    #print(f"Duplicated timestamps in index: {info['duplicated_timestamps_in_index']}")
    #print(f"Missing total cells: {info['missing_total_cells']}")
    #print(f"Missing by col: {info['missing_by_col']}")
    #print(f"Rows/day stats: {info['rows_per_day_stats']}")
    print(f"Time-of-day range (minutes): {info['time_of_day_minutes_range']}")
    print(f"UTC note: {info['utc_note']}")
    if info["utc_range_if_applicable"] is not None:
        print(f"UTC range: {info['utc_range_if_applicable'][0]}  ->  {info['utc_range_if_applicable'][1]}")


In [9]:
info = mnq_dataset_info(mnq_raw, name="mnq_raw", tz_assume_if_naive=None, day_def="trading")
print_mnq_dataset_info(info)

Dataset: mnq_raw
Shape: (1902941, 5)
Columns: ['open', 'high', 'low', 'close', 'volume']
Index: DatetimeIndex | TZ: tz-naive (sin zona horaria)
Datetime min/max: 2019-12-23 03:01:00  ->  2025-06-16 00:03:00
First/Last day: 2019-12-23  ->  2025-06-16
Total days (trading): 1756
Time-of-day range (minutes): {'min_minute_of_day': 0, 'max_minute_of_day': 1439}
UTC note: El índice es tz-naive; no se puede asegurar si está en UTC sin suposiciones.


In [10]:
def count_total_days(df: pd.DataFrame) -> int:
    """
    Cuenta la cantidad total de días distintos presentes en un DataFrame
    con índice de tipo DatetimeIndex.

    Parámetros
    ----------
    df : pandas.DataFrame
        DataFrame indexado por datetime.

    Retorna
    -------
    int
        Número total de días distintos en el dataset.
    """
    # Extraer la fecha (YYYY-MM-DD) del índice y contar valores únicos
    total_days = df.index.normalize().nunique()

    return int(total_days)

In [11]:
total_days_raw = count_total_days(mnq_raw)
print(f"Total days (RAW): {total_days_raw}")

Total days (RAW): 1756


# **2. Filtrado de días no hábiles y horario bursátil**

## 2.1. Filtrado de fines de semana y feriados bursátiles estadounidenses

Es necesario filtrar del conjunto de datos aquellas filas correspondientes a sábados, domingos y feriados bursátiles. Para ello, se utilizará la librería pandas_market_calendars, que permite identificar los días hábiles de operación según el calendario oficial del NASDAQ.

La función implementada filtra un DataFrame con índice de tipo DatetimeIndex, conservando únicamente aquellas filas cuya fecha coincida con un día hábil del mercado. La marca temporal completa (fecha y hora) se mantiene sin modificaciones.

In [12]:
def filter_nasdaq_trading_days(df):
    """
    Filtra un DataFrame para conservar únicamente los días hábiles
    de negociación del mercado NASDAQ, en función de su DatetimeIndex.

    Parámetros
    ----------
    df : pandas.DataFrame
        DataFrame indexado por fechas y horas (DatetimeIndex).

    Retorna
    -------
    pandas.DataFrame
        DataFrame que contiene solo las filas correspondientes
        a días oficiales de trading del NASDAQ.
    """
    # Inicializar el calendario oficial del mercado NASDAQ
    nasdaq_calendar = mcal.get_calendar("NASDAQ")

    # Determinar el rango de fechas a partir del índice del DataFrame
    start_date = df.index.min().date()
    end_date = df.index.max().date()

    # Obtener el cronograma oficial de días hábiles del NASDAQ
    trading_days = nasdaq_calendar.schedule(
        start_date=start_date,
        end_date=end_date
    ).index.date

    # Filtrar el DataFrame manteniendo solo fechas válidas de trading
    filtered_df = df[df.index.normalize().isin(trading_days)]

    return filtered_df

In [13]:
mnq_intraday = filter_nasdaq_trading_days (mnq_raw)

In [14]:
trading_days = count_total_days(mnq_intraday)
print(f"Trading days: {trading_days}")

Trading days: 1370


## 2.2. Filtrado de horario de operación de mercado de New York (09:00 a 16:00) con pre mercado, desde las 06:30

Dado que los timestamps del índice (DatetimeIndex) provienen de archivos .txt sin información de zona horaria, es necesario indicar explícitamente a pandas que dichos valores están en formato UTC.

Una vez establecido el timezone, se procede a convertir los timestamps desde UTC a la hora local del mercado estadounidense (zona US/Eastern), correspondiente a los horarios de operación del NASDAQ/NYSE. Esta conversión se realiza teniendo en cuenta automáticamente los ajustes por horario de verano o invierno.

In [15]:
def configure_timezone(df, from_tz="UTC", to_tz="America/New_York"):
    """
    Asegura que el índice del DataFrame tenga definida la zona horaria
    `from_tz` y luego lo convierte a la zona horaria `to_tz`.

    Parámetros
    ----------
    df : pandas.DataFrame
        DataFrame con índice de tipo DatetimeIndex.
    from_tz : str, opcional
        Zona horaria de origen a asignar si el índice no tiene tz (por defecto "UTC").
    to_tz : str, opcional
        Zona horaria destino a la que se convertirá el índice
        (por defecto "America/New_York").

    Retorna
    -------
    pandas.DataFrame
        DataFrame con el índice correctamente localizado y convertido.
    """
    # Verificar si el índice no tiene información de zona horaria
    if df.index.tz is None:
        # Asignar la zona horaria de origen sin modificar los timestamps
        df.index = df.index.tz_localize(from_tz)

    # Convertir el índice a la zona horaria destino
    df.index = df.index.tz_convert(to_tz)

    return df

In [16]:
mnq_intraday = configure_timezone(mnq_intraday)

In [17]:
mnq_intraday

,open,high,low,close,volume
datetime,,,,,
2019-12-22 22:01:00-05:00,8718.50,8718.75,8718.50,8718.50,9
2019-12-22 22:02:00-05:00,8718.25,8718.25,8718.00,8718.25,14
2019-12-22 22:03:00-05:00,8718.25,8718.50,8718.00,8718.25,74
2019-12-22 22:04:00-05:00,8718.25,8719.00,8718.25,8718.50,10
2019-12-22 22:05:00-05:00,8718.50,8719.00,8718.50,8719.00,6
...,...,...,...,...,...
2025-06-13 17:00:00-04:00,21651.25,21659.50,21649.75,21654.00,610
2025-06-15 20:00:00-04:00,21686.50,21691.75,21686.00,21687.25,103
2025-06-15 20:01:00-04:00,21687.25,21694.00,21684.50,21688.00,266


La siguiente función selecciona únicamente las muestras que se encuentran dentro del horario regular de operación bursátil del NASDAQ.

Filtra un DataFrame cuyo índice es de tipo DatetimeIndex, conservando solo aquellas filas cuya marca temporal se encuentre entre las 09:30 y 16:00 horas (US/Eastern), correspondientes al horario de negociación estándar en días hábiles de mercado.

Particularmente, decido agregar una hora de pre mercado, desde las 06:30AM.

In [18]:
def filter_nasdaq_trading_hours(df, start_time: str, end_time: str):
    """
    Filtra un DataFrame para conservar únicamente las filas que se
    encuentren dentro del horario de negociación del NASDAQ.

    Parámetros
    ----------
    df : pandas.DataFrame
        DataFrame indexado por DatetimeIndex.
    start_time : str
        Hora de inicio del mercado (por ejemplo, "08:00:00").
    end_time : str
        Hora de cierre del mercado (por ejemplo, "16:00:00").

    Retorna
    -------
    pandas.DataFrame
        DataFrame filtrado dentro del rango horario especificado.
    """
    # Filtrar filas que estén dentro del rango horario de trading
    filtered_df = df.between_time(start_time, end_time)

    return filtered_df

In [19]:
mnq_intraday = filter_nasdaq_trading_hours(mnq_intraday, '06:30:00', '16:00:00' )

In [20]:
trading_session_days = count_total_days(mnq_intraday)
print(f"Trading session days (06:30 to 16:00): {trading_session_days }")

Trading session days (06:30 to 16:00): 1367


In [21]:
info_intraday = mnq_dataset_info(mnq_intraday, name="mnq_intraday", tz_assume_if_naive="America/New_York", day_def="trading")
print_mnq_dataset_info(info_intraday)

Dataset: mnq_intraday
Shape: (769753, 5)
Columns: ['open', 'high', 'low', 'close', 'volume']
Index: DatetimeIndex | TZ: America/New_York
Datetime min/max: 2019-12-23 06:30:00-05:00  ->  2025-06-13 16:00:00-04:00
First/Last day: 2019-12-23  ->  2025-06-13
Total days (trading): 1367
Time-of-day range (minutes): {'min_minute_of_day': 390, 'max_minute_of_day': 960}
UTC note: El índice está tz-aware; el horario UTC es inequívoco.
UTC range: 2019-12-23 11:30:00+00:00  ->  2025-06-13 20:00:00+00:00


# **3. Análisis de registros diarios**

Es necesario verificar que todos los días del conjunto de datos contengan la misma cantidad de registros y que estos sean consecutivos, es decir, que no falte ningún minuto dentro de cada jornada.

La función analizar_registros_por_dia permite realizar este control sobre un DataFrame con índice de tipo datetime. La función contabiliza la cantidad de registros por día e imprime una tabla resumen que indica cuántos días presentan una determinada cantidad de registros. Esto resulta útil para identificar inconsistencias, como días incompletos o interrupciones en la frecuencia temporal esperada.

In [22]:
def analyze_daily_record_counts(df: pd.DataFrame) -> tuple[pd.Series, int]:
    """
    Analiza la cantidad de registros por día en un DataFrame con índice datetime.

    Imprime:
    - La cantidad de registros correspondiente a un día completo.
    - El número y porcentaje de días con menos registros que dicho valor.

    Retorna:
    -------
    tuple[pandas.Series, int]
        - Serie con el conteo de registros por día.
        - Cantidad de registros correspondiente a un día completo
          (valor más frecuente).
    """
    # Contar la cantidad de registros por día
    daily_counts = df.groupby(df.index.date).size()

    # Determinar el número de registros de un día completo (moda)
    full_day_records = daily_counts.mode().iloc[0]
    print(f"Cantidad de registros en un día completo: {full_day_records}")

    # Calcular el porcentaje de días con registros incompletos
    total_days = len(daily_counts)
    incomplete_days = (daily_counts < full_day_records).sum()
    incomplete_percentage = (incomplete_days / total_days) * 100

    print(
        f"Días con menos de {full_day_records} registros: "
        f"{incomplete_days} de {total_days} ({incomplete_percentage:.2f}%)"
    )

    return daily_counts, full_day_records

In [23]:
#resumen, registros_dia_completo = analyze_daily_record_counts(mnq_intraday)
daily_record_counts, full_day_record_count = analyze_daily_record_counts(mnq_intraday)

Cantidad de registros en un día completo: 571
Días con menos de 571 registros: 64 de 1367 (4.68%)


Como podemos observar en la tabla, la gran mayoría de días tienen `571` muestras. Y representan más del 95% del total de los datos.


## **3.1. Filtrado de días incompletos**

La siguiente función encuentra los indices de las fechas con registros incompletos:

In [24]:
def find_incomplete_trading_dates(
    df: pd.DataFrame,
    expected_records: int,
    gap_minutes: int = 1,
) -> list[pd.Timestamp]:
    """
    Identifica los días que presentan menos registros de los esperados
    o irregularidades en la secuencia temporal de los datos.

    Parámetros
    ----------
    df : pandas.DataFrame
        DataFrame con índice de tipo DatetimeIndex.
    expected_records : int
        Cantidad esperada de registros por día.
    gap_minutes : int, opcional
        Intervalo esperado entre registros consecutivos, en minutos
        (por defecto 1 minuto).

    Retorna
    -------
    list[pandas.Timestamp]
        Lista de fechas que presentan registros incompletos
        o irregularidades temporales.
    """
    df = df.copy()

    # Calcular la diferencia temporal entre registros consecutivos
    df["time_diff"] = df.index.to_series().diff()
    expected_time_diff = pd.Timedelta(minutes=gap_minutes)

    # Conteo de registros por día
    daily_counts = df.groupby(df.index.date).size()

    problematic_dates = []

    # Analizar cada día de forma independiente
    for date, group in df.groupby(df.index.date):
        time_diffs = group["time_diff"].iloc[1:]
        has_irregular_gaps = (time_diffs != expected_time_diff).any()
        record_count = daily_counts[date]

        if record_count < expected_records or has_irregular_gaps:
            problematic_dates.append(date)

    return problematic_dates

Elimino las fechas con registros incompletos:

In [25]:
def remove_incomplete_trading_days(
    df: pd.DataFrame,
    expected_records: int,
) -> pd.DataFrame:
    """
    Elimina del DataFrame los días que presentan registros incompletos
    o irregularidades temporales, y vuelve a analizar los registros diarios.

    Parámetros
    ----------
    df : pandas.DataFrame
        DataFrame con índice de tipo DatetimeIndex.
    expected_records : int
        Cantidad esperada de registros por día.

    Retorna
    -------
    pandas.DataFrame
        DataFrame filtrado, conteniendo únicamente días completos
        y sin irregularidades temporales.
    """
    # Identificar fechas problemáticas (días incompletos o con gaps)
    problematic_dates = find_incomplete_trading_dates(
        df=df,
        expected_records=expected_records,
    )

    # Filtrar el DataFrame eliminando las fechas con problemas
    cleaned_df = df[
        ~df.index.to_series().dt.date.isin(problematic_dates)
    ]

    # Reanalizar la cantidad de registros por día tras la limpieza
    analyze_daily_record_counts(cleaned_df)

    return cleaned_df

In [26]:
#df_mnq = eliminar_fechas_incompletas(df_mnq, registros_dia_completo)
mnq_intraday = remove_incomplete_trading_days(
    df = mnq_intraday,
    expected_records = full_day_record_count,
)

Cantidad de registros en un día completo: 571
Días con menos de 571 registros: 0 de 1303 (0.00%)


In [27]:
trading_session_complete_days = count_total_days(mnq_intraday)
print(f"Trading session complete days : {trading_session_complete_days }")

Trading session complete days : 1303


In [28]:
info_intraday = mnq_dataset_info(mnq_intraday, name="mnq_intraday", tz_assume_if_naive="America/New_York", day_def="trading")
print_mnq_dataset_info(info_intraday)

Dataset: mnq_intraday
Shape: (744013, 5)
Columns: ['open', 'high', 'low', 'close', 'volume']
Index: DatetimeIndex | TZ: America/New_York
Datetime min/max: 2019-12-23 06:30:00-05:00  ->  2025-06-13 16:00:00-04:00
First/Last day: 2019-12-23  ->  2025-06-13
Total days (trading): 1303
Time-of-day range (minutes): {'min_minute_of_day': 390, 'max_minute_of_day': 960}
UTC note: El índice está tz-aware; el horario UTC es inequívoco.
UTC range: 2019-12-23 11:30:00+00:00  ->  2025-06-13 20:00:00+00:00


#**4. Verificación de continuidad temporal minuto a minuto**

Es necesario verificar que los registros correspondientes a un mismo día estén dispuestos de forma consecutiva, con una separación exacta de un minuto entre cada muestra.

In [29]:
def detect_time_gaps(
    df: pd.DataFrame,
    gap_minutes: int = 1,
) -> list[pd.DatetimeIndex]:
    """
    Verifica la existencia de saltos temporales mayores al intervalo esperado
    entre registros consecutivos dentro de cada día.

    Se omite el primer registro de cada jornada, ya que no tiene referencia previa.

    Parámetros
    ----------
    df : pandas.DataFrame
        DataFrame con índice de tipo DatetimeIndex.
    gap_minutes : int, opcional
        Intervalo temporal esperado entre registros consecutivos, en minutos
        (por defecto 1 minuto).

    Retorna
    -------
    list[pandas.DatetimeIndex]
        Lista de índices donde se detectaron diferencias temporales
        distintas al intervalo esperado.
    """
    df = df.copy()

    # Calcular diferencias temporales entre registros consecutivos
    df["time_diff"] = df.index.to_series().diff()
    expected_time_diff = pd.Timedelta(minutes=gap_minutes)

    problematic_indices = []

    # Analizar cada día de manera independiente
    for date, group in df.groupby(df.index.date):
        time_diffs = group["time_diff"].iloc[1:]
        irregular_indices = time_diffs[time_diffs != expected_time_diff].index

        if not irregular_indices.empty:
            problematic_indices.append(irregular_indices)

    if problematic_indices:
        print(
            f"Se encontraron problemas en {len(problematic_indices)} "
            "bloques diarios con diferencias temporales irregulares.\n"
        )

        # Conteo de registros por día
        daily_counts = df.groupby(df.index.date).size()

        for indices in problematic_indices:
            idx = indices[0]
            time_diff = df.loc[idx, "time_diff"]
            date = idx.date()
            record_count = daily_counts[date]

            print(
                f"\t{idx} -> Diferencia: {time_diff} "
                f"| # Registros del día: {record_count}"
            )
    else:
        print(
            "No se encontraron problemas: todas las muestras "
            "son consecutivas minuto a minuto."
        )

    return problematic_indices

In [30]:
detect_time_gaps(mnq_intraday)

No se encontraron problemas: todas las muestras son consecutivas minuto a minuto.


[]

#**5. Búsqueda de NaNs**


In [31]:
# Verificar si existen NaNs en el dataset
has_nans = mnq_intraday.isna().any().any()

# Cantidad total de NaNs
total_nans = int(mnq_intraday.isna().sum().sum())

# Cantidad de NaNs por columna
nans_by_column = mnq_intraday.isna().sum()

print("=== NaN Check: mnq_intraday ===")
print(f"¿Existen NaNs en el dataset?: {has_nans}")
print(f"Cantidad total de NaNs: {total_nans}\n")

print("NaNs por columna:")
#display(nans_by_column.to_frame(name="NaN count"))
print(nans_by_column)


=== NaN Check: mnq_intraday ===
¿Existen NaNs en el dataset?: False
Cantidad total de NaNs: 0

NaNs por columna:
open      0
high      0
low       0
close     0
volume    0
dtype: int64


# **6. Definición de regímenes intradía y flags de sesión**

## **6.1. Definición**

Se definen los regímenes intradía mediante flags binarios, sin solapamientos y cubriendo todo el día de negociación.

---

**1. Regímenes definidos**

Los regímenes se definen en hora de Nueva York, utilizando intervalos del tipo **[inicio, fin)** (incluye el inicio, excluye el fin):

- **Pre-market** (`is_premarket`):  
  08:30 ≤ t < 09:30

- **Opening (apertura)** (`is_opening`):  
  09:30 ≤ t < 11:00

- **Regular** (`is_regular`):  
  11:00 ≤ t < 15:00

- **Closing (cierre)** (`is_closing`):  
  15:00 ≤ t < 16:00

- **Closed** (`is_closed`):  
  t < 08:30  ó  t ≥ 16:00

---

**2. Consistencia de flags**

Para cada timestamp debe cumplirse:

- Exactamente **un único flag activo**:
  `is_premarket + is_opening + is_regular + is_closing + is_closed = 1`


Esto garantiza una segmentación exhaustiva y mutuamente excluyente del régimen de mercado.

---

**3. Conversión usando `minute_of_day`**

Los cortes horarios pueden implementarse de forma determinística utilizando `minute_of_day`:

- 08:30 → 510  
- 09:30 → 570  
- 11:00 → 660  
- 15:00 → 900  
- 16:00 → 960  

---

**4. Uso en modelos de forecasting**

Todos los flags de régimen (`is_premarket`, `is_opening`, `is_regular`, `is_closing`, `is_closed`) deben ser tratados como **known future covariates**, ya que dependen únicamente del calendario y no del comportamiento del mercado.  
Este tipo de variables resulta especialmente útil en modelos multi-horizonte, como el Temporal Fusion Transformer, aunque su definición es compatible con cualquier enfoque de forecasting.

---

**Conclusión**

La segmentación propuesta define de forma clara los distintos regímenes intradía del mercado, evita ambigüedades y permite su utilización consistente en la evaluación comparativa de distintos modelos (naive, regresiones lineales, TCN, MLP, LSTM y Transformers), sin condicionar el análisis a un único tipo de arquitectura.


## **6.2. Agregado de columna `minute_of_day`**

In [32]:
def add_minute_of_day(
    df: pd.DataFrame,
    *,
    col_name: str = "minute_of_day",
) -> pd.DataFrame:
    """
    Agrega la columna `minute_of_day` al DataFrame.

    Definición:
      minute_of_day = hour * 60 + minute
      rango: [0, 1439]

    Requisitos:
      - df.index debe ser DatetimeIndex
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError("El DataFrame debe tener un DatetimeIndex.")

    out = df.copy()
    out[col_name] = out.index.hour * 60 + out.index.minute
    return out


In [33]:
def add_column_date(df):
    # Asegurar que el índice esté en formato datetime
    df.index = pd.to_datetime(df.index)

    # Crear una nueva columna 'date' con la fecha extraída del índice
    df['date'] = df.index.date

    # Reordenar columnas: 'date', 'time_str', y luego el resto
    cols = ['date'] + [col for col in df.columns if col not in ['date']]

    df = df[cols]

    return df

In [34]:
mnq_intraday = add_minute_of_day(mnq_intraday)
mnq_intraday = add_column_date(mnq_intraday)

## **6.3. Agregado de flags**

In [35]:
import pandas as pd

def add_market_regime_flags(mnq_intraday: pd.DataFrame) -> pd.DataFrame:
    """
    Agrega flags binarios de régimen intradía basados en minute_of_day.

    Regímenes (hora NY, intervalos [inicio, fin)):
      - premarket : 08:30 ≤ t < 09:30
      - opening   : 09:30 ≤ t < 11:00
      - regular   : 11:00 ≤ t < 15:00
      - closing   : 15:00 ≤ t < 16:00
      - closed    : t < 08:30  o  t ≥ 16:00
    """

    df = mnq_intraday.copy()

    m = df["minute_of_day"]

    df["is_premarket"] = ((m >= 510) & (m < 570)).astype(int)
    df["is_opening"]   = ((m >= 570) & (m < 660)).astype(int)
    df["is_regular"]   = ((m >= 660) & (m < 900)).astype(int)
    df["is_closing"]   = ((m >= 900) & (m < 960)).astype(int)

    # Closed = todo lo que queda fuera de 08:30–16:00
    df["is_closed"] = (1 - (
        df["is_premarket"]
        + df["is_opening"]
        + df["is_regular"]
        + df["is_closing"]
    )).astype(int)

    return df


In [36]:
mnq_intraday = add_market_regime_flags(mnq_intraday)


In [37]:
mnq_intraday.head(182)

,date,open,high,low,close,volume,minute_of_day,is_premarket,is_opening,is_regular,is_closing,is_closed
datetime,,,,,,,,,,,,
2019-12-23 06:30:00-05:00,2019-12-23,8727.75,8728.00,8727.75,8727.75,7,390,0,0,0,0,1
2019-12-23 06:31:00-05:00,2019-12-23,8727.50,8727.75,8726.50,8726.50,89,391,0,0,0,0,1
2019-12-23 06:32:00-05:00,2019-12-23,8726.50,8726.50,8725.25,8725.25,34,392,0,0,0,0,1
2019-12-23 06:33:00-05:00,2019-12-23,8725.50,8726.25,8724.75,8726.00,53,393,0,0,0,0,1
2019-12-23 06:34:00-05:00,2019-12-23,8726.00,8726.00,8726.00,8726.00,3,394,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...
2019-12-23 09:27:00-05:00,2019-12-23,8731.75,8732.00,8731.25,8732.00,21,567,1,0,0,0,0
2019-12-23 09:28:00-05:00,2019-12-23,8732.00,8733.50,8732.00,8732.25,74,568,1,0,0,0,0
2019-12-23 09:29:00-05:00,2019-12-23,8731.75,8734.50,8730.50,8734.25,114,569,1,0,0,0,0


# **7. Agregado de flags por día**

## **7.1. Justificación del uso de flags binarios para el día hábil**

El día de la semana es una variable de calendario que introduce efectos sistemáticos en el comportamiento intradía del mercado (volumen, volatilidad, direccionalidad). Para representarla como feature, se adopta el uso de **flags binarios por día hábil** en lugar de una codificación ordinal única.

**Ventajas del enfoque con flags binarios**

- **Evita relaciones ordinales artificiales**  
  Una codificación numérica (`day_of_week = 0…4`) introduce un orden implícito entre los días (por ejemplo, “viernes > lunes”), lo cual no tiene interpretación económica. Los flags binarios tratan cada día como una categoría independiente.

- **Mejor compatibilidad con modelos lineales**  
  En modelos como Lasso y Ridge, los flags permiten aprender desplazamientos de régimen específicos por día (efectos aditivos), sin forzar una relación lineal entre categorías.

- **Mayor expresividad en modelos no lineales**  
  En MLP, TCN, LSTM y Transformers, los flags facilitan el aprendizaje de interacciones del tipo *patrón de precios × día de la semana*, sin ambigüedad semántica.

- **Consistencia entre arquitecturas**  
  El mismo esquema de features puede utilizarse de forma idéntica en modelos simples y complejos, permitiendo comparaciones justas sin modificar el input.

- **Naturaleza determinística**  
  Los días hábiles dependen únicamente del calendario, por lo que los flags pueden tratarse como **known future covariates**, una propiedad especialmente aprovechable en modelos multi-horizonte como el TFT.

**Conclusión**

El uso de flags binarios por día hábil proporciona una representación más neutra, interpretable y robusta que una codificación ordinal, y resulta adecuada para la evaluación comparativa de distintos modelos de forecasting intradía.


## **7.2. Implementación**

In [38]:
import pandas as pd

def add_weekday_flags(mnq_intraday: pd.DataFrame) -> pd.DataFrame:
    """
    Agrega flags binarios por día hábil (lunes a viernes)
    a partir del índice DatetimeIndex (hora NY).

    Flags creados:
      - is_mon
      - is_tue
      - is_wed
      - is_thu
      - is_fri
    """

    df = mnq_intraday.copy()

    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError("El DataFrame debe tener un DatetimeIndex.")

    dow = df.index.dayofweek  # lunes=0, ..., domingo=6

    df["is_mon"] = (dow == 0).astype(int)
    df["is_tue"] = (dow == 1).astype(int)
    df["is_wed"] = (dow == 2).astype(int)
    df["is_thu"] = (dow == 3).astype(int)
    df["is_fri"] = (dow == 4).astype(int)

    return df


In [39]:
mnq_intraday = add_weekday_flags(mnq_intraday)


In [40]:
mnq_intraday

,date,open,high,low,close,volume,minute_of_day,is_premarket,is_opening,is_regular,is_closing,is_closed,is_mon,is_tue,is_wed,is_thu,is_fri
datetime,,,,,,,,,,,,,,,,,
2019-12-23 06:30:00-05:00,2019-12-23,8727.75,8728.00,8727.75,8727.75,7,390,0,0,0,0,1,1,0,0,0,0
2019-12-23 06:31:00-05:00,2019-12-23,8727.50,8727.75,8726.50,8726.50,89,391,0,0,0,0,1,1,0,0,0,0
2019-12-23 06:32:00-05:00,2019-12-23,8726.50,8726.50,8725.25,8725.25,34,392,0,0,0,0,1,1,0,0,0,0
2019-12-23 06:33:00-05:00,2019-12-23,8725.50,8726.25,8724.75,8726.00,53,393,0,0,0,0,1,1,0,0,0,0
2019-12-23 06:34:00-05:00,2019-12-23,8726.00,8726.00,8726.00,8726.00,3,394,0,0,0,0,1,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-06-13 15:56:00-04:00,2025-06-13,21624.50,21635.00,21613.50,21617.50,3251,956,0,0,0,1,0,0,0,0,0,1
2025-06-13 15:57:00-04:00,2025-06-13,21616.50,21635.25,21615.75,21623.75,2201,957,0,0,0,1,0,0,0,0,0,1
2025-06-13 15:58:00-04:00,2025-06-13,21623.25,21632.75,21616.50,21621.75,1859,958,0,0,0,1,0,0,0,0,0,1


#**8. Guardado de dataset final y resumen**



## **8.1. Guardado de parquet `mnq_intraday`**


In [41]:
# Verificar si el archivo ya existe
if os.path.exists(OUT_PARQUET):
    print(f"Archivo encontrado en disco: {OUT_PARQUET}")
    mnq_intraday = pd.read_parquet(OUT_PARQUET)
    print("Dataset cargado desde Drive.")
else:
    print("No se encontró el archivo en Drive. Guardando nuevo dataset...")
    os.makedirs(os.path.dirname(OUT_PARQUET), exist_ok=True)
    mnq_intraday.to_parquet(OUT_PARQUET, index=True)
    print(f"Dataset guardado en: {OUT_PARQUET}")

No se encontró el archivo en Drive. Guardando nuevo dataset...
Dataset guardado en: /content/drive/MyDrive/neural_profit/data/processed/mnq_intraday.parquet


## **8.2. Generación y guardado `stage_01_data_prep_summary.json`**


In [42]:
info_intraday_final = mnq_dataset_info(mnq_intraday, name="mnq_intraday", tz_assume_if_naive="America/New_York", day_def="trading")
print_mnq_dataset_info(info_intraday_final)

Dataset: mnq_intraday
Shape: (744013, 17)
Columns: ['date', 'open', 'high', 'low', 'close', 'volume', 'minute_of_day', 'is_premarket', 'is_opening', 'is_regular', 'is_closing', 'is_closed', 'is_mon', 'is_tue', 'is_wed', 'is_thu', 'is_fri']
Index: DatetimeIndex | TZ: America/New_York
Datetime min/max: 2019-12-23 06:30:00-05:00  ->  2025-06-13 16:00:00-04:00
First/Last day: 2019-12-23  ->  2025-06-13
Total days (trading): 1303
Time-of-day range (minutes): {'min_minute_of_day': 390, 'max_minute_of_day': 960}
UTC note: El índice está tz-aware; el horario UTC es inequívoco.
UTC range: 2019-12-23 11:30:00+00:00  ->  2025-06-13 20:00:00+00:00


In [43]:
OUT_SUMMARY

PosixPath('/content/drive/MyDrive/neural_profit/reports/stage_01_dataset_prep_summary.json')

In [44]:
from pathlib import Path
import json

# ------------------------------------------------------------
# Guardado del summary del stage_01 en formato JSON
# ------------------------------------------------------------

# Crear la carpeta destino si no existe
OUT_SUMMARY.parent.mkdir(parents=True, exist_ok=True)

# Guardar el resumen del dataset en formato JSON
with OUT_SUMMARY.open("w", encoding="utf-8") as f:
    json.dump(
        info_intraday_final,
        f,
        indent=2,
        ensure_ascii=False
    )


# **9. Alineación con libro ML**


**1. Estructura general del dataset**

- **Formato**: parquet.  
- **Índice temporal**: `datetime` con timezone.  
- **Columnas**: `open`, `high`, `low`, `close`, `volume` (OHLCV).  
- **Tamaño**: ~744.000 registros, consistente con un dataset intradía multianual.

    **Conclusión**: estructura sólida y estándar para modelado ML intradía.

**2. Orden temporal y continuidad**

- Los timestamps se encuentran **estrictamente ordenados**.  
- No se detectan **duplicados por minuto**.  
- El índice temporal es **monótono creciente**.

    **Conclusión**: no existen problemas de orden ni desalineación temporal.

**3. Separación por jornada**

- Los datos están **segmentados por jornada bursátil**.  
- No se mezclan observaciones entre días.  
- Cada jornada respeta una **ventana intradía consistente**.

    Este punto es crítico para el modelado intradía y se encuentra correctamente implementado.

**4. Horario de sesión**

- El dataset base cubre el intervalo **06:30–16:00 (ET)**.  
- Se aplica un **filtrado posterior** para definir ventanas específicas de uso  
  (por ejemplo, *gestation* / *trading window*).

**5. Coherencia OHLCV**

- Se cumple la relación `high ≥ open/close ≥ low`.  
- Los volúmenes son **no negativos**.  
- No se observan valores aberrantes evidentes.

    **Conclusión**: datos de mercado consistentes y sanos.

**6. Base para targets H = 60 / H = 90**

- El dataset presenta **profundidad intradía suficiente** para construir targets a horizontes H = 60 y H = 90 minutos.  
- Los últimos minutos de cada jornada deben descartarse al construir los targets, de acuerdo con la definición temporal del problema.

---

**Diagnóstico global**

La preparación de los datos es **correcta, limpia y metodológicamente sólida**.  
No se identifican señales de *look-ahead bias*, mezcla de jornadas ni inconsistencias estructurales.
